# Занятие 7. Работа с датами и агрегации

> **Цель занятия:** научиться работать с датами и временем в Pandas — превращать текст
> в `Timestamp`, доставать из даты нужные компоненты (год, месяц, день недели, час),
> считать интервалы между событиями, а также собирать сводные отчёты через `groupby`,
> `pivot_table` и `crosstab`.

**Что будет:**
1. **Типы `Timestamp` и `Timedelta`** — как Pandas хранит даты и промежутки времени;
2. **`pd.to_datetime()` и аксессор `.dt`** — как достать из даты год, месяц, день недели, час;
3. **Интервалы между событиями** — вычитание дат и работа с `Timedelta`;
4. **Split-Apply-Combine и `.groupby()`** — базовый механизм агрегаций;
5. **Сводные таблицы** — `pivot_table` и `crosstab` как инструмент отчётности.


In [100]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## Загружаем данные про заказы интернет-магазина

Мы будем работать с таблицей заказов онлайн-магазина за 2023–2024 годы —
1500 строк. У каждого заказа есть дата оформления, дата доставки, город,
категория товара, способ оплаты и сумма.


In [101]:
df = pd.read_csv('data/orders.csv', sep=';')
print('Размер таблицы:', df.shape)
df.head()

Размер таблицы: (1500, 9)


,order_id,created_at,delivered_at,customer_id,city,category,payment,amount,items
0,570,2024-02-03 17:52,2024-02-05 12:27,1313,Москва,Книги,Карта,2052,2
1,431,2024-10-21 03:19,2024-10-21 12:27,1356,Москва,Косметика,СБП,2637,3
2,80,2024-10-18 02:37,2024-10-24 10:32,1050,Самара,Книги,СБП,431,1
3,336,2023-10-15 23:52,2023-10-19 07:41,1236,Самара,Одежда,Карта,2925,4
4,719,2023-06-23 04:37,2023-06-26 17:27,1300,Самара,Электроника,Карта,14247,1


Посмотрим на **типы столбцов** — это важно перед работой с датами:


In [102]:
df.dtypes

order_id         int64
created_at      object
delivered_at    object
customer_id      int64
city            object
category        object
payment         object
amount           int64
items            int64
dtype: object

**Вопрос:** столбцы `created_at` и `delivered_at` сейчас имеют тип `object` — обычный текст. Что именно мы **не сможем** с ними сделать, пока не преобразуем их в настоящие даты?

---
## Часть 1. Типы `Timestamp` и `Timedelta`

В Pandas есть два основных «временных» типа:

- **`Timestamp`** — момент времени (дата и, при необходимости, время). Аналог `datetime`
  из стандартной библиотеки, но заточенный под Pandas.
- **`Timedelta`** — **промежуток** времени: «3 дня 5 часов», «12 минут». Получается,
  например, как разность двух `Timestamp`.

```
Timestamp   →   момент:      2024-05-01 12:30
Timedelta   →   промежуток:  3 days 05:00:00
```

Оба типа — «умные»: они понимают арифметику, умеют доставать компоненты и работают
столбцом сразу для всей таблицы.

### 1.1. Как создать `Timestamp` вручную


In [103]:
# один момент времени
t = pd.Timestamp('2024-05-01 12:30')
t

Timestamp('2024-05-01 12:30:00')

In [104]:
# из компонентов
pd.Timestamp(year=2024, month=5, day=1, hour=12, minute=30)

Timestamp('2024-05-01 12:30:00')

### 1.2. Как получить `Timedelta`

Самый частый способ — **вычесть одну дату из другой**:


In [105]:
t1 = pd.Timestamp('2024-05-01 12:30')
t2 = pd.Timestamp('2024-05-05 09:00')
delta = t2 - t1
delta

Timedelta('3 days 20:30:00')

**Вывод:** результат — `Timedelta`, промежуток «3 дня 20 часов 30 минут».
С ним можно работать как с числом: сравнивать, суммировать, брать среднее.

Ещё `Timedelta` можно создать «напрямую», задав промежуток словами:


In [106]:
pd.Timedelta(days=2, hours=6)

Timedelta('2 days 06:00:00')

---
## Часть 2. `pd.to_datetime()` и аксессор `.dt`

### 2.1. Превращаем текст в даты

Наши столбцы `created_at` и `delivered_at` пока текстовые. Чтобы Pandas
воспринимал их как даты, нужно **явно преобразовать** через `pd.to_datetime()`:


In [107]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['delivered_at'] = pd.to_datetime(df['delivered_at'])
df.dtypes

order_id                 int64
created_at      datetime64[ns]
delivered_at    datetime64[ns]
customer_id              int64
city                    object
category                object
payment                 object
amount                   int64
items                    int64
dtype: object

**Вывод:** теперь оба столбца имеют тип `datetime64[ns]` — то есть массив
`Timestamp`. Это ключевой шаг: без него не сработают ни `.dt`-аксессор, ни вычисление
интервалов.

> **На заметку:** если в столбце попадётся «плохая» дата — например, опечатка
> `2024-13-40`, — то `pd.to_datetime()` по умолчанию завершится с ошибкой. Чтобы такие
> значения не ломали расчёт, добавляют аргумент `errors='coerce'`: тогда они
> превращаются в `NaT` — «пропуск» для дат, аналог `NaN`.

### 2.2. Достаём компоненты даты через `.dt`

У столбца-даты есть особый аксессор **`.dt`** — через него можно достать любую часть
даты сразу для всей колонки, без циклов. Это как `.str` для строк.


In [108]:
# год
df['created_at'].dt.year.head()

0    2024
1    2024
2    2024
3    2023
4    2023
Name: created_at, dtype: int32

In [109]:
# месяц (число от 1 до 12)
df['created_at'].dt.month.head()

0     2
1    10
2    10
3    10
4     6
Name: created_at, dtype: int32

In [110]:
# день недели (0 = понедельник, 6 = воскресенье)
df['created_at'].dt.dayofweek.head()

0    5
1    0
2    4
3    6
4    4
Name: created_at, dtype: int32

In [111]:
# час
df['created_at'].dt.hour.head()

0    17
1     3
2     2
3    23
4     4
Name: created_at, dtype: int32

> **Попробуйте сами:** как бы вы получили название дня недели («Monday», «Tuesday» и т. д.)
> для каждого заказа — по аналогии с примерами выше?


In [137]:
# ...

**Полезные атрибуты `.dt`:**

| Атрибут | Что возвращает |
|---|---|
| `.dt.year` | год (число) |
| `.dt.month` | номер месяца (1–12) |
| `.dt.day` | число месяца (1–31) |
| `.dt.hour` / `.dt.minute` | час / минута |
| `.dt.dayofweek` | день недели (0 = Пн, 6 = Вс) |
| `.dt.day_name()` / `.dt.month_name()` | название дня / месяца |
| `.dt.date` | только дата, без времени |
| `.dt.quarter` | номер квартала (1–4) |

### 2.3. Добавим полезные признаки

Часто в анализе полезно **сразу создать колонки** с годом, месяцем и т. п.,
чтобы потом группировать по ним:


In [113]:
df['год'] = df['created_at'].dt.year
df['месяц'] = df['created_at'].dt.month
df['день_недели'] = df['created_at'].dt.day_name()
df['час'] = df['created_at'].dt.hour

df[['created_at', 'год', 'месяц', 'день_недели', 'час']].head()

,created_at,год,месяц,день_недели,час
0,2024-02-03 17:52:00,2024,2,Saturday,17
1,2024-10-21 03:19:00,2024,10,Monday,3
2,2024-10-18 02:37:00,2024,10,Friday,2
3,2023-10-15 23:52:00,2023,10,Sunday,23
4,2023-06-23 04:37:00,2023,6,Friday,4


Ещё один частый вопрос к данным — **в какие часы дня заказов больше всего**. Чтобы
посчитать, сколько раз встречается каждое значение в столбце, используют метод
**`value_counts()`**:


In [114]:
df['час'].value_counts()

час
7     88
12    82
5     74
3     73
4     71
18    68
23    67
20    67
2     64
16    64
19    64
21    62
17    62
14    61
13    61
6     58
10    57
1     57
11    56
8     55
0     55
9     51
15    44
22    39
Name: count, dtype: int64

Здесь пригодился столбец `час`, а посчитать повторения помогает метод **`value_counts()`**.
Он проходит по столбцу и считает, **сколько раз встречается каждое уникальное значение**,
после чего возвращает `Series`: в индексе — само значение (здесь час), рядом — сколько
раз оно встретилось. По умолчанию строки идут **по убыванию частоты** — сверху самый
популярный час, то есть пиковое время.

Полезные параметры `value_counts()`:

| Параметр | Что делает |
|---|---|
| `normalize=True` | вместо количеств возвращает **доли** (в сумме дают 1) |
| `ascending=True` | сортировать по **возрастанию** частоты, а не по убыванию |
| `dropna=False` | учитывать пропуски (`NaN`) как отдельное значение |

Например, доли заказов по часам:


In [115]:
df['час'].value_counts(normalize=True).round(3).head()

час
7     0.059
12    0.055
5     0.049
3     0.049
4     0.047
Name: proportion, dtype: float64

**Вывод:** то же распределение, но в долях — какая часть всех заказов приходится
на каждый час.

Часто результат удобнее видеть **по порядку самого часа** (0, 1, 2, …), а не по частоте.
Тогда сортируют не по значениям, а по **индексу** — методом `.sort_index()`:


In [116]:
df['час'].value_counts().sort_index()

час
0     55
1     57
2     64
3     73
4     71
5     74
6     58
7     88
8     55
9     51
10    57
11    56
12    82
13    61
14    61
15    44
16    64
17    62
18    68
19    64
20    67
21    62
22    39
23    67
Name: count, dtype: int64

Здесь встретились **две разные сортировки** — их важно не путать:

| Метод | По чему сортирует | Что даёт здесь |
|---|---|---|
| `.sort_index()` | по **индексу** | часы по порядку: 0, 1, 2, … |
| `.sort_values()` | по **значениям** | часы по числу заказов |

У обоих есть параметр `ascending`: `True` — по возрастанию (значение по умолчанию),
`False` — по убыванию.

**Вывод:** `value_counts()` быстро считает частоты, а `.sort_index()` и `.sort_values()`
дают полный контроль над порядком строк в результате.


---
## Часть 3. Интервалы между событиями

Когда обе колонки — даты, их **можно вычитать**. Результат — столбец `Timedelta`.

### 3.1. Считаем время доставки


In [117]:
df['доставка'] = df['delivered_at'] - df['created_at']
df[['created_at', 'delivered_at', 'доставка']].head()

,created_at,delivered_at,доставка
0,2024-02-03 17:52:00,2024-02-05 12:27:00,1 days 18:35:00
1,2024-10-21 03:19:00,2024-10-21 12:27:00,0 days 09:08:00
2,2024-10-18 02:37:00,2024-10-24 10:32:00,6 days 07:55:00
3,2023-10-15 23:52:00,2023-10-19 07:41:00,3 days 07:49:00
4,2023-06-23 04:37:00,2023-06-26 17:27:00,3 days 12:50:00


**Вывод:** столбец `доставка` — это `Timedelta`. У недоставленных заказов
(`delivered_at` пустой) значение — `NaT`.

### 3.2. Как достать «дни» или «часы» из интервала

Часто `Timedelta` неудобен для расчётов — его лучше перевести в число. Для этого
есть два способа:


In [118]:
# способ 1: атрибут .dt.days — только дни (целое число), часы отбрасываются
df['доставка'].dt.days.head()

0    1.0
1    0.0
2    6.0
3    3.0
4    3.0
Name: доставка, dtype: float64

In [119]:
# способ 2: .dt.total_seconds() — общее число секунд, дальше можно делить
(df['доставка'].dt.total_seconds() / 86400).head()   # в днях, с дробной частью

0    1.774306
1    0.380556
2    6.329861
3    3.325694
4    3.534722
Name: доставка, dtype: float64

**Вопрос:** доставка заняла ровно полтора суток. Что вернёт `.dt.days`, а что — `.dt.total_seconds() / 86400`? В каком случае какой способ уместнее?

### 3.3. Средняя скорость доставки


In [120]:
df['доставка_дней'] = df['доставка'].dt.total_seconds() / 86400
df['доставка_дней'].mean().round(2)

2.35

**Вывод:** среднее время доставки — около 2.35 дня. Недоставленные заказы, у которых
интервал получился `NaT`, Pandas при `.mean()` **игнорирует автоматически** — так же,
как обычные `NaN`. Поэтому среднее считается по доставленным заказам, и вручную удалять
пропуски не нужно.

### 3.4. Фильтрация по интервалу

`Timedelta` умеет сравниваться — как обычное число:


In [121]:
# заказы, которые ехали дольше 5 дней
long = df[df['доставка'] > pd.Timedelta(days=5)]
print('Долгих доставок:', len(long))
long[['city', 'category', 'доставка']].head()

Долгих доставок: 74


,city,category,доставка
2,Самара,Книги,6 days 07:55:00
24,Самара,Электроника,5 days 02:42:00
36,Новосибирск,Косметика,5 days 03:46:00
52,Новосибирск,Дом,6 days 15:05:00
61,Новосибирск,Электроника,6 days 07:17:00


---
## Часть 4. Split-Apply-Combine и `.groupby()`

Теперь — центральная тема анализа: **как посчитать что-то по группам**. Например:
средний чек **по каждому городу**, число заказов **по каждой категории**, медианное
время доставки **по способу оплаты**.

Механизм у всех таких задач один и тот же — **Split-Apply-Combine**:

```
┌──────────────┐    Split     ┌──────────────┐
│   таблица    │ ───────────▶ │ группы строк │   (разбили по значению столбца)
└──────────────┘              └──────────────┘
                                     │
                                     │ Apply    (посчитали что-то в каждой группе)
                                     ▼
                              ┌──────────────┐
                              │ по 1 числу   │
                              │  на группу   │
                              └──────────────┘
                                     │
                                     │ Combine  (собрали обратно в таблицу)
                                     ▼
                              ┌──────────────┐
                              │  результат   │
                              └──────────────┘
```

В Pandas за всё это отвечает **`.groupby()`**.

Важно понимать: сам по себе вызов `df.groupby('колонка')` **ещё ничего не считает** —
он лишь размечает, какие строки в какую группу попадут. Результат появляется только
когда к группам применяют функцию: `.size()`, `.mean()`, `.sum()` и другие.

### 4.1. Простой `groupby`

Сколько заказов в каждой категории?


In [122]:
df.groupby('category').size()

category
Дом            243
Книги          272
Косметика      222
Одежда         401
Спорт          140
Электроника    222
dtype: int64

**Вывод:** `groupby('category')` разбил таблицу на группы по категориям,
а `.size()` посчитал число строк в каждой. Получилась `Series`: индекс — категория,
значение — число заказов.

### 4.2. Среднее по группе

Средний чек по городам:


In [123]:
df.groupby('city')['amount'].mean().round().sort_values(ascending=False)

city
Самара             4953.0
Новосибирск        4642.0
Ростов-на-Дону     4321.0
Москва             4226.0
Нижний Новгород    4110.0
Санкт-Петербург    3754.0
Казань             3471.0
Екатеринбург       3234.0
Name: amount, dtype: float64

**Что произошло:**
- `.groupby('city')` — разбили строки по значению `city`;
- `['amount']` — интересует только этот столбец;
- `.mean()` — считаем среднее в каждой группе;
- `.sort_values(...)` — упорядочили результат по убыванию.

Получили готовую сводку: где средний чек выше, где ниже.

> **Попробуйте сами:** как бы вы посчитали **суммарную выручку** по каждой категории?
> Логика та же — заменить `.mean()` на подходящую функцию.


In [124]:
df.groupby('category')['amount'].sum().sort_values(ascending=False)

category
Электроника    2725205
Одежда         1446274
Спорт           625723
Дом             610043
Косметика       427059
Книги           245730
Name: amount, dtype: int64

**Вопрос:** суммарная выручка оказалась самой большой у категории `Электроника`, хотя заказов в ней не больше, чем в остальных. Чем это можно объяснить?

In [125]:
df.groupby('category')['amount'].agg(['count', 'mean', 'sum']).round(1)

,count,mean,sum
category,,,
Дом,243,2510.5,610043
Книги,272,903.4,245730
Косметика,222,1923.7,427059
Одежда,401,3606.7,1446274
Спорт,140,4469.4,625723
Электроника,222,12275.7,2725205


**Вывод:** одной строчкой получили полноценный отчёт по категориям:
сколько заказов, средний чек, суммарная выручка.



**Частые функции агрегации** — их вызывают напрямую (`.mean()`) или передают списком
в `.agg([...])`:

| Функция | Что считает |
|---|---|
| `'count'` | сколько непустых значений в группе |
| `'sum'` | сумму |
| `'mean'` / `'median'` | среднее / медиану |
| `'min'` / `'max'` | минимум / максимум |
| `'std'` | стандартное отклонение (разброс значений) |
| `'nunique'` | сколько **разных** значений в группе |
| `'first'` / `'last'` | первое / последнее значение в группе |


### 4.4. Разным столбцам — свои функции

`.agg([...])` со списком применяет одни и те же функции ко **всей** выбранной колонке.
Но часто для разных столбцов нужны **разные** метрики: по заказам — их число, по сумме —
средний чек и общая выручка. Тогда каждому столбцу отчёта задают пару
`(столбец, функция)` — это называют **именованной агрегацией**:


In [126]:
df.groupby('city').agg(
    заказов=('order_id', 'count'),
    средний_чек=('amount', 'mean'),
    выручка=('amount', 'sum'),
).round(1)

,заказов,средний_чек,выручка
city,,,
Екатеринбург,120,3234.4,388132
Казань,138,3470.8,478966
Москва,523,4226.5,2210442
Нижний Новгород,96,4110.0,394558
Новосибирск,123,4642.3,571001
Ростов-на-Дону,98,4320.7,423431
Самара,87,4952.5,430871
Санкт-Петербург,315,3754.4,1182633


**Что произошло:** слева от `=` — имя нового столбца в отчёте, справа — пара
«из какого столбца считать» и «какой функцией». Столбец `amount` здесь использован
дважды — для среднего чека и для выручки — с разными именами результата.

**Вывод:** именованная агрегация — самый гибкий и читаемый способ: в одном `groupby`
собирается отчёт, где каждый показатель посчитан своей функцией из своего столбца.

> Есть и более короткая запись через словарь — `df.groupby('city').agg({'order_id': 'count',
> 'amount': 'mean'})`, — но она не даёт задать имена столбцов результата.


### 4.5. Группировка по нескольким колонкам

В `groupby` можно передать **список столбцов** — тогда группы будут определяться
их сочетанием. Например, город и категория:


In [127]:
df.groupby(['city', 'category'])['amount'].mean().round().head(12)

city          category   
Екатеринбург  Дом             2967.0
              Книги            852.0
              Косметика       1742.0
              Одежда          3633.0
              Спорт           4397.0
              Электроника    10747.0
Казань        Дом             2344.0
              Книги            907.0
              Косметика       1876.0
              Одежда          3964.0
              Спорт           4279.0
              Электроника    12140.0
Name: amount, dtype: float64

**Вывод:** получилась `Series` с **иерархическим индексом** — сначала город,
потом категория. Смотреть удобно, но с такой структурой не всегда просто работать.
Гораздо нагляднее это же представить как **сводную таблицу** — этим займёмся в части 5.

> **Подумайте:** что изменится в коде выше, если нам нужен не средний чек, а **медиана**?
> Достаточно ли поменять одно слово?


In [138]:
# df.groupby(['city', 'category'])['amount']. ...

Логика группировки при этом не меняется — различается только функция агрегации в конце цепочки. В этом и удобство `.groupby()`.


### 4.6. От иерархического индекса — к обычной таблице

Когда группируем по **нескольким** колонкам, ключи (`city` и `category`) уходят
в **иерархический индекс** (`MultiIndex`): они лежат не обычными столбцами, а «вложены»
в индекс результата. С такой формой не всегда удобно работать дальше — фильтровать,
соединять с другими таблицами, сохранять в файл.

Чтобы вернуть **плоскую таблицу**, где ключи снова станут обычными столбцами, есть
два способа.


In [129]:
# способ 1: .reset_index() — уносит уровни индекса обратно в столбцы
df.groupby(['city', 'category'])['amount'].mean().round().reset_index().head()

,city,category,amount
0,Екатеринбург,Дом,2967.0
1,Екатеринбург,Книги,852.0
2,Екатеринбург,Косметика,1742.0
3,Екатеринбург,Одежда,3633.0
4,Екатеринбург,Спорт,4397.0


**Вывод:** теперь `city` и `category` — обычные столбцы, а средний чек лежит
в столбце `amount`. Это привычный плоский `DataFrame`.


In [130]:
# способ 2: тот же результат сразу — параметр as_index=False прямо в groupby
df.groupby(['city', 'category'], as_index=False)['amount'].mean().round().head()

,city,category,amount
0,Екатеринбург,Дом,2967.0
1,Екатеринбург,Книги,852.0
2,Екатеринбург,Косметика,1742.0
3,Екатеринбург,Одежда,3633.0
4,Екатеринбург,Спорт,4397.0


**Вывод:** `as_index=False` говорит `groupby` **не** уносить ключи группировки в индекс.
Результат тот же, что и после `.reset_index()`, — выбирают тот вариант, что короче
в конкретном месте.


---
## Часть 5. Сводные таблицы: `pivot_table` и `crosstab`

Сводные таблицы — это **тот же `groupby`**, только результат сразу разворачивается
в удобную двумерную форму: одна переменная по строкам, другая — по столбцам.
Это стандартный формат отчёта — как в Excel.

### 5.1. `pd.pivot_table()` — сводная с числами

Хотим отчёт: **средний чек** по городам (строки) и категориям (столбцы).


In [131]:
pd.pivot_table(
    df,
    values='amount',       # что считаем
    index='city',          # что по строкам
    columns='category',    # что по столбцам
    aggfunc='mean',        # какой функцией агрегируем
).round()

category,Дом,Книги,Косметика,Одежда,Спорт,Электроника
city,,,,,,
Екатеринбург,2967.0,852.0,1742.0,3633.0,4397.0,10747.0
Казань,2344.0,907.0,1876.0,3964.0,4279.0,12140.0
Москва,2415.0,937.0,2045.0,3640.0,4372.0,11991.0
Нижний Новгород,2794.0,800.0,2002.0,3768.0,4015.0,13492.0
Новосибирск,1834.0,912.0,1588.0,3402.0,5014.0,13299.0
Ростов-на-Дону,2728.0,828.0,1696.0,3263.0,4537.0,13286.0
Самара,2331.0,736.0,2121.0,3616.0,5075.0,13720.0
Санкт-Петербург,2769.0,935.0,1919.0,3535.0,4686.0,11248.0


**Что произошло:**
- `values='amount'` — считаем среднее суммы;
- `index='city'` — города по строкам;
- `columns='category'` — категории по столбцам;
- `aggfunc='mean'` — агрегирующая функция.

Получилась таблица «город × категория», в клетках — средний чек. Это готовый
отчёт, который можно показать бизнесу.

> **А если** нам нужен не средний чек, а **сумма выручки** — как изменится вызов?
> Попробуйте описать по аналогии с примером выше.


In [139]:
# pd.pivot_table(
# ...
# ).fillna(0).astype(int)

**Вопрос:** Почему в клетке на пересечении конкретного города и категории вообще может не оказаться значения?

In [133]:
pd.pivot_table(
    df,
    values='amount',
    index='city',
    aggfunc=['count', 'mean', 'sum'],
).round(1).head()

,count,mean,sum
,amount,amount,amount
city,,,
Екатеринбург,120,3234.4,388132
Казань,138,3470.8,478966
Москва,523,4226.5,2210442
Нижний Новгород,96,4110.0,394558
Новосибирск,123,4642.3,571001


**Вывод:** одна таблица — сразу число заказов, средний чек и выручка по каждому городу.

### 5.3. `pd.crosstab()` — сводная с частотами

Если нужна **не сумма/среднее числа**, а просто **сколько раз встречается сочетание**
двух категорий — есть более короткая функция **`pd.crosstab()`**.

Например: сколько заказов в каждом городе оплачено каждым способом?


In [134]:
pd.crosstab(df['city'], df['payment'])

payment,Карта,Наличные,СБП
city,,,
Екатеринбург,71,10,39
Казань,69,13,56
Москва,276,57,190
Нижний Новгород,56,10,30
Новосибирск,60,15,48
Ростов-на-Дону,56,5,37
Самара,46,7,34
Санкт-Петербург,187,32,96


In [135]:
# доли внутри каждой строки (по городам) — сумма по строке = 1
(pd.crosstab(df['city'], df['payment'], normalize='index') * 100).round(1)

payment,Карта,Наличные,СБП
city,,,
Екатеринбург,59.2,8.3,32.5
Казань,50.0,9.4,40.6
Москва,52.8,10.9,36.3
Нижний Новгород,58.3,10.4,31.2
Новосибирск,48.8,12.2,39.0
Ростов-на-Дону,57.1,5.1,37.8
Самара,52.9,8.0,39.1
Санкт-Петербург,59.4,10.2,30.5


**Вывод:** видно, какой процент заказов в каждом городе оплачивают картой,
СБП или наличными. Значения `normalize`:

- `'index'` — доли по строкам (каждая строка суммируется в 100%);
- `'columns'` — доли по столбцам;
- `'all'` — доли от общего итога.

### 5.5. Соединяем даты и агрегации — заказы по месяцам


In [136]:
# заказов по месяцам и категориям
pd.pivot_table(
    df,
    values='order_id',
    index='месяц',
    columns='category',
    aggfunc='count',
).fillna(0).astype(int)

category,Дом,Книги,Косметика,Одежда,Спорт,Электроника
месяц,,,,,,
1,20,20,13,39,13,25
2,22,24,20,34,15,10
3,13,22,20,48,16,16
4,22,14,17,34,8,14
5,23,14,17,36,13,17
6,18,27,23,34,11,20
7,26,28,12,31,8,19
8,25,22,21,32,17,11
9,26,12,24,30,5,30


**Вывод:** видна сезонность — по месяцам можно заметить провалы и пики.
Это одна из самых частых задач бизнес-отчётности: **как менялся показатель во времени
по разрезам**.

### 5.6. Когда что использовать?

| Инструмент | Когда удобно |
|---|---|
| **`.groupby()`** | Результат — колонка / серия. Гибкая агрегация, дальше — своя логика. |
| **`pivot_table`** | Нужен **двумерный отчёт**: строки — одно, столбцы — другое, в клетках — число. |
| **`crosstab`** | Просто **посчитать частоты** сочетаний двух категорий (или доли через `normalize`). |

---
## Итог занятия

Сегодня научились:
- различать **`Timestamp`** (момент времени) и **`Timedelta`** (промежуток);
- превращать текст в даты через **`pd.to_datetime()`** и доставать компоненты через
  аксессор **`.dt`** (`.dt.year`, `.dt.month`, `.dt.dayofweek`, `.dt.hour`,
  `.dt.day_name()` и др.);
- считать **интервалы** между событиями как разность двух дат и переводить их в дни
  или секунды через `.dt.days` / `.dt.total_seconds()`;
- применять механизм **Split-Apply-Combine** через **`.groupby()`** — с одним или
  несколькими столбцами, с одной или несколькими функциями агрегации;
- строить **сводные таблицы** через **`pd.pivot_table()`** и частотные таблицы
  через **`pd.crosstab()`** — со счётом, средними, суммами, долями.

**Главная мысль:** даты — это **полноценный тип данных**, а не строки, и с ними
работает целый набор специальных инструментов. А `groupby` и сводные таблицы —
основной способ **превратить сырые строки в понятный отчёт**.
